# 03 Advanced Chunking Strategies

**Real-World Scenario**: Technical Documentation & Cloud API Manual Retrieval System.

This notebook demonstrates **Parent-Child Hierarchical Chunking** (retrieving small 128-token child chunks for high vector search precision, returning 1024-token parent context blocks to the LLM), completed by an **End-to-End GenAI LLM Call**.

## Step 1: Environment Setup

In [ ]:
import os
from dotenv import load_dotenv

# Load API keys from repository root .env
load_dotenv()

# Create hidden vector database directory for local index persistence
os.makedirs(".vectordb", exist_ok=True)

print("=== Repository Environment Setup ===")
print("OPENAI_API_KEY present:", bool(os.getenv("OPENAI_API_KEY")))
print("GEMINI_API_KEY present:", bool(os.getenv("GEMINI_API_KEY")))
print("Hidden DB Directory Path:", os.path.abspath(".vectordb"))


## Step 2: Parent-Child Hierarchical Chunking Implementation

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

parent_doc = """AWS Infrastructure Security & Observability Specification:
Section 1: AWS KMS Encryption Key Policy
All S3 buckets storing customer PII must enforce server-side encryption with AWS KMS customer-managed keys (SSE-KMS). Key rotation must occur automatically every 365 days.

Section 2: Datadog Telemetry Log Retention
Datadog log forwarder agents deployed on EKS clusters must scrub social security numbers and API secrets before transmission. Telemetry logs are retained for 30 days in hot storage and archived to S3 Glacier for 7 years.

Section 3: AWS SNS Notification Alerts
Alert notifications for CPU utilization exceeding 90% for 5 consecutive minutes are routed to AWS SNS topic 'infra-alerts-prod' and trigger PagerDuty calls to on-call engineers."""

# Parent & Child Splitters
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=350, chunk_overlap=0)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=0)

parents = parent_splitter.split_text(parent_doc)
print(f"=== Parent-Child Split Completed ===")
print(f"Parent Chunks Count: {len(parents)}")


## Step 3: Indexing Child Chunks Linked to Parent Store in Hidden `.vectordb/` Directory

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

if os.getenv("OPENAI_API_KEY"):
    child_texts = []
    child_metadatas = []
    
    for p_idx, p_text in enumerate(parents):
        children = child_splitter.split_text(p_text)
        for c_text in children:
            child_texts.append(c_text)
            child_metadatas.append({"parent_idx": p_idx, "parent_text": p_text})
            
    vectorstore = FAISS.from_texts(child_texts, OpenAIEmbeddings(), metadatas=child_metadatas)
    save_path = os.path.join(".vectordb", "03_parent_child_index")
    vectorstore.save_local(save_path)
    print(f"Indexed {len(child_texts)} child chunks linked to {len(parents)} parent documents.")
    print(f"Saved to hidden folder: {save_path}")


## Step 4: Executing Complete GenAI Parent Context LLM Synthesis Call

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

if os.getenv("OPENAI_API_KEY"):
    save_path = os.path.join(".vectordb", "03_parent_child_index")
    embeddings = OpenAIEmbeddings()
    vectorstore = FAISS.load_local(save_path, embeddings, allow_dangerous_deserialization=True)
    
    query = "What is the KMS key rotation policy?"
    child_matches = vectorstore.similarity_search(query, k=1)
    
    parent_context = child_matches[0].metadata["parent_text"]
    prompt = ChatPromptTemplate.from_template("Answer question strictly using full parent context:\nParent Context:\n{context}\n\nQuestion: {question}\nAnswer:")
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)
    
    response = llm.invoke(prompt.format(context=parent_context, question=query))
    print("=== Complete Parent-Child GenAI Response ===")
    print("Matched Child Chunk Text:", child_matches[0].page_content)
    print("Injected Parent Context Length:", len(parent_context), "characters")
    print("\nLLM Grounded Response:")
    print(response.content)
